# r-z23 models: v1 baseline and v2
Both versions are defined in `src/amlop_ca1/train_rz23.py`, which also logs them to MLflow. This notebook only diagnoses v1 and checks each v2 change on the validation set.

In [1]:
import sys
import logging

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier

sys.path.append("../src")
from amlop_ca1 import cleaning, train_rz23
from amlop_ca1.features import REQUIRED_COLUMNS, make_feature_pipeline

logging.getLogger("amlop_ca1").setLevel(logging.ERROR)
DATA_DIR = "../data"
COLUMNS = list(REQUIRED_COLUMNS)

In [2]:
customers, orders, labels = cleaning.load_batch(DATA_DIR, "base")
df = cleaning.clean_and_merge(customers, orders, labels)
train, valid, test, train_end, valid_end = train_rz23.split_by_date(df)

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name}: {len(part)} rows, {part['order_date'].min().date()} to {part['order_date'].max().date()}, fraud rate {part['is_fraud'].mean():.3f}")

train: 660772 rows, 2022-11-12 to 2023-03-14, fraud rate 0.101
valid: 144978 rows, 2023-03-15 to 2023-04-13, fraud rate 0.136
test: 144121 rows, 2023-04-14 to 2023-05-11, fraud rate 0.124


## v1 baseline

In [3]:
v1 = train_rz23.build_model("v1", train["is_fraud"], "cpu")
v1.fit(train[COLUMNS], train["is_fraud"])
v1_prob = v1.predict_proba(valid[COLUMNS])[:, 1]
pd.Series(train_rz23.scores(valid["is_fraud"], v1_prob, 0.5, "valid")).round(3)

valid_precision    0.678
valid_recall       0.070
valid_f1           0.127
valid_pr_auc       0.450
valid_roc_auc      0.814
dtype: float64

In [4]:
pd.crosstab(valid["is_fraud"], (v1_prob >= 0.5).astype(int), rownames=["actual"], colnames=["predicted"])

predicted,0,1
actual,,
0,124582,656
1,18358,1382


### Which features does v1 use?

In [5]:
importance = pd.Series(v1["model"].feature_importances_, index=v1["features"].get_feature_names_out())
importance.sort_values(ascending=False).head(10).round(3)

categorical__country_code_MY                       0.420
categorical__country_code_TH                       0.136
categorical__country_code_PH                       0.124
categorical__country_code_PK                       0.062
categorical__country_code_BD                       0.060
categorical__collect_type_delivery                 0.032
categorical__payment_method_generic creditcard     0.028
categorical__payment_method_credit card            0.028
numeric__order_value                               0.020
categorical__payment_method_payment on delivery    0.017
dtype: float32

### Does fraud drift over time?

In [6]:
df.groupby([df["order_date"].dt.to_period("M"), "country_code"])["is_fraud"].mean().unstack().round(3)

country_code,BD,MY,PH,PK,TH
order_date,,,,,
2022-11,0.147,0.039,0.105,0.200,0.050
2022-12,0.270,0.038,0.101,0.183,0.062
2023-01,0.294,0.036,0.091,0.203,0.057
2023-02,0.287,0.035,0.086,0.225,0.062
2023-03,0.314,0.040,0.072,0.337,0.059
2023-04,0.352,0.043,0.063,0.298,0.063
2023-05,0.441,0.050,0.078,0.249,0.065


### Does order_month help on a month the model has not seen?
The first v1 used order_month, which features.py has since dropped. This check adds it back by hand. Validation shares March with training, so it stays inside the training period: train on earlier months, score the next one.

In [7]:
for start, end in [("2023-01-01", "2023-02-01"), ("2023-02-01", "2023-03-01")]:
    early = train[train["order_date"] < start]
    later = train[(train["order_date"] >= start) & (train["order_date"] < end)]
    features = make_feature_pipeline().fit(early[COLUMNS])
    early_X = features.transform(early[COLUMNS]).toarray()
    later_X = features.transform(later[COLUMNS]).toarray()
    early_month = early["order_date"].dt.month.to_numpy().reshape(-1, 1)
    later_month = later["order_date"].dt.month.to_numpy().reshape(-1, 1)

    without_month = XGBClassifier(**train_rz23.PARAMS["v1"], tree_method="hist", random_state=42).fit(early_X, early["is_fraud"])
    with_month = XGBClassifier(**train_rz23.PARAMS["v1"], tree_method="hist", random_state=42).fit(np.hstack([early_X, early_month]), early["is_fraud"])
    print(start[:7], "with month:", round(average_precision_score(later["is_fraud"], with_month.predict_proba(np.hstack([later_X, later_month]))[:, 1]), 3),
          " without month:", round(average_precision_score(later["is_fraud"], without_month.predict_proba(later_X)[:, 1]), 3))

2023-01 with month: 0.285  without month: 0.311


2023-02 with month: 0.334  without month: 0.332


### Does weighting the fraud class help?

In [8]:
weight = (train["is_fraud"] == 0).sum() / train["is_fraud"].sum()
weighted = train_rz23.build_model("v1", train["is_fraud"], "cpu").set_params(model__scale_pos_weight=weight)
weighted.fit(train[COLUMNS], train["is_fraud"])
print("scale_pos_weight:", round(weight, 2))
pd.DataFrame({
    "v1": train_rz23.scores(valid["is_fraud"], v1_prob, 0.5, "valid"),
    "v1 + weight": train_rz23.scores(valid["is_fraud"], weighted.predict_proba(valid[COLUMNS])[:, 1], 0.5, "valid"),
}).round(3)

scale_pos_weight: 8.88


,v1,v1 + weight
valid_precision,0.678,0.290
valid_recall,0.070,0.793
valid_f1,0.127,0.424
valid_pr_auc,0.450,0.452
valid_roc_auc,0.814,0.813


### Do the tree settings matter?

In [9]:
for depth in [4, 6, 8]:
    model = train_rz23.build_model("v2", train["is_fraud"], "cpu").set_params(model__max_depth=depth)
    model.fit(train[COLUMNS], train["is_fraud"])
    print("max_depth", depth, "valid pr_auc:", round(average_precision_score(valid["is_fraud"], model.predict_proba(valid[COLUMNS])[:, 1]), 4))

max_depth 4 valid pr_auc: 0.4554


max_depth 6 valid pr_auc: 0.4517


max_depth 8 valid pr_auc: 0.4449


## v1 vs v2
v2 picks its threshold on validation; test is only scored.

In [10]:
v2 = train_rz23.build_model("v2", train["is_fraud"], "cpu")
v2.fit(train[COLUMNS], train["is_fraud"])
v2_valid_prob = v2.predict_proba(valid[COLUMNS])[:, 1]
threshold = train_rz23.best_threshold(valid["is_fraud"], v2_valid_prob)
print("v2 threshold:", round(threshold, 3))

results = {}
for name, model, cut in [("v1", v1, 0.5), ("v2", v2, threshold)]:
    results[name] = {
        **train_rz23.scores(valid["is_fraud"], model.predict_proba(valid[COLUMNS])[:, 1], cut, "valid"),
        **train_rz23.scores(test["is_fraud"], model.predict_proba(test[COLUMNS])[:, 1], cut, "test"),
    }
pd.DataFrame(results).round(3)

v2 threshold: 0.62


,v1,v2
valid_precision,0.678,0.427
valid_recall,0.070,0.613
valid_f1,0.127,0.503
valid_pr_auc,0.450,0.455
valid_roc_auc,0.814,0.816
test_precision,0.612,0.341
test_recall,0.071,0.619
test_f1,0.127,0.440
test_pr_auc,0.388,0.390
test_roc_auc,0.786,0.788


## Customer fields (open team question)
Customer history is excluded because it has no as-of time. How often is the customer's first order later than the order itself, and what would the fields add?

In [11]:
print("first order after this order:", round((df["first_order_datetime"] > df["order_date"]).mean(), 4))
print("customer rows per customer:", customers.duplicated(["country_code", "customer_id"]).sum(), "repeats")

first order after this order: 0.0151
customer rows per customer: 0 repeats


In [12]:
history = ["num_orders_last_50days", "num_cancelled_orders_last_50days", "num_refund_orders_last_50days",
           "total_payment_last_50days", "num_associated_customers"]
features = make_feature_pipeline().fit(train[COLUMNS])

def with_history(part):
    return np.hstack([features.transform(part[COLUMNS]).toarray(), part[history].to_numpy(dtype=float)])

model = XGBClassifier(**train_rz23.PARAMS["v1"], tree_method="hist", random_state=42)
model.fit(with_history(train), train["is_fraud"])
print("valid pr_auc with history:", round(average_precision_score(valid["is_fraud"], model.predict_proba(with_history(valid))[:, 1]), 3))
print("test pr_auc with history:", round(average_precision_score(test["is_fraud"], model.predict_proba(with_history(test))[:, 1]), 3))

valid pr_auc with history: 0.518


test pr_auc with history: 0.497


## Decisions

| v1 problem | Evidence | v2 change |
|---|---|---|
| Misses most fraud | Recall 0.07 at 0.5; 18,358 of 19,740 validation fraud orders missed | `scale_pos_weight` = non-fraud / fraud in training (8.9) |
| 0.5 is an arbitrary cut-off | Weighting alone moves recall to 0.79 but precision to 0.29 | Threshold with the best validation F1 (0.62), saved in the model |
| Deeper trees overfit slightly | max_depth 4 / 6 / 8: 0.455 / 0.452 / 0.445 validation PR-AUC | max_depth 4 |

v2 lifts test F1 from 0.13 to 0.44 and recall from 0.07 to 0.62, with test PR-AUC 0.390 against 0.388.

`order_month` was in the first v1. On an unseen month it adds nothing (Feb: 0.334 vs 0.332) or hurts (Jan: 0.285 vs 0.311), and new batches always bring unseen months, so it was dropped from features.py for both versions.

Fraud rates drift by country (BD 0.15 to 0.44), so both versions need retraining on each new batch.

Customer history raises test PR-AUC to 0.497 but has no as-of time (1.5% of orders come before the customer's first order). It stays out until the team agrees it is available at prediction time.